In [ ]:
# --- ensure repo-root cwd so data/ paths resolve from anywhere ---
import os, sys
from pathlib import Path
_r = Path.cwd()
while not (_r / 'requirements.txt').exists() and _r != _r.parent:
    _r = _r.parent
os.chdir(_r); sys.path.insert(0, str(_r / 'src'))

# SoH by coulomb counting — Mahindra vehicles (intellicar table)

The **intellicar** table carries real signed pack `current` (A) and `batteryVoltage` (V) at
sub-second resolution, so we can do **true coulomb counting** — the method originally requested,
impossible on the mahindra/euler OEM feeds (no current).

**Method.** Charge throughput is `Q = ∫ I·dt` (Ah). Within a continuous-logging *session*
(samples close in time), we accumulate `Q` and regress it against `soc`. The slope `dQ/dSoC`
(Ah per % SoC) × 100 is the **usable pack capacity** in Ah — independent of the current sign
convention and robust to noise. Capacity per month, normalised to early-life = 100%, then a
**monotonic non-increasing envelope** (SoH never rises).

Data: `data/intellicar_extracted/` (oldest 10 Mahindra Treo/Zor Grand by odometer).

## 1. Load and clean

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow.dataset as ds
import matplotlib.pyplot as plt

EXTRACTED = Path("data/mahindra/_archive/intellicar_oldest10")
SOH_OUT = Path("data/mahindra/soh"); SOH_OUT.mkdir(parents=True, exist_ok=True)

SESSION_GAP_S = 30.0      # >30 s between samples starts a new logging session
MIN_SOC_SPAN = 20.0       # a session must span >=20% SoC to estimate capacity
MIN_SESSION_ROWS = 40     # and have enough samples for a stable regression
CAP_BOUNDS_AH = (40.0, 400.0)   # plausible usable pack capacity for a Treo-class EV

df = ds.dataset(EXTRACTED, format="parquet").to_table().to_pandas()
print(f"Loaded {len(df):,} rows, {df['vin'].nunique()} VINs, models: {df['model'].dropna().unique().tolist()}")

df["t"] = pd.to_datetime(pd.to_numeric(df["eventAt"], errors="coerce"), unit="ms", errors="coerce")
for c in ["soc", "current", "batteryVoltage", "odometer"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")
df = df.dropna(subset=["vin", "t", "soc", "current"])
df = df[(df["soc"] >= 0) & (df["soc"] <= 100)]
df = df.sort_values(["vin", "t"]).reset_index(drop=True)
print(f"After cleaning: {len(df):,} rows")
print("current range:", df["current"].min(), "->", df["current"].max())

## 2. Coulomb integration into sessions

Per VIN, split into sessions on time gaps > `SESSION_GAP_S`. Within a session, integrate
`current` over `dt` to a cumulative charge `Q` (Ah). Sign convention is irrelevant — we use
the magnitude of the `Q`-vs-`soc` slope.

In [ ]:
out = []
for vin, g in df.groupby("vin"):
    g = g.sort_values("t")
    dt = g["t"].diff().dt.total_seconds().to_numpy()
    new_session = ~np.isfinite(dt) | (dt > SESSION_GAP_S) | (dt <= 0)
    session_id = np.cumsum(new_session)
    dt_clip = np.where(new_session, 0.0, dt)
    cur = g["current"].to_numpy()
    # trapezoidal-ish: average current across the step
    cur_avg = np.where(new_session, 0.0, (cur + np.roll(cur, 1)) / 2.0)
    dQ = cur_avg * dt_clip / 3600.0     # Ah per step
    gg = g.assign(session=session_id, dQ=dQ)
    gg["Qcum"] = gg.groupby("session")["dQ"].cumsum()
    out.append(gg[["vin", "t", "soc", "current", "batteryVoltage", "odometer", "session", "Qcum"]])
samples = pd.concat(out, ignore_index=True)
print(f"{samples['session'].nunique():,} (vin,session) groups across {samples['vin'].nunique()} VINs")

## 3. Per-session capacity = |slope of Q vs SoC| × 100

For each session spanning enough SoC with enough samples, fit `Qcum ~ soc` and take
`|slope| × 100` as the usable capacity (Ah).

In [ ]:
lo, hi = CAP_BOUNDS_AH
recs = []
for (vin, sess), s in samples.groupby(["vin", "session"]):
    if len(s) < MIN_SESSION_ROWS:
        continue
    soc = s["soc"].to_numpy()
    if (soc.max() - soc.min()) < MIN_SOC_SPAN:
        continue
    slope = np.polyfit(soc, s["Qcum"].to_numpy(), 1)[0]
    cap = abs(slope) * 100.0
    if lo <= cap <= hi:
        recs.append({"vin": vin, "t": s["t"].iloc[0], "capacity_ah": cap,
                     "soc_span": soc.max() - soc.min(), "n": len(s)})
caps = pd.DataFrame(recs)
caps["month"] = caps["t"].dt.to_period("M").dt.to_timestamp()
print(f"{len(caps)} valid capacity sessions across {caps['vin'].nunique()} VINs")
print(caps["capacity_ah"].describe().round(1))

## 4. Monthly SoH (normalised + monotonic)

In [ ]:
monthly = (caps.groupby(["vin", "month"])
                .agg(capacity_ah=("capacity_ah", "median"), n_sessions=("capacity_ah", "size"))
                .reset_index())

def to_soh(g):
    g = g.sort_values("month").copy()
    base = g["capacity_ah"].iloc[0]
    g["soh_raw"] = 100.0 * g["capacity_ah"] / base
    g["soh"] = g["soh_raw"].cummin()       # monotonic non-increasing
    return g

soh = monthly.groupby("vin", group_keys=False).apply(to_soh)
print(f"SoH points: {len(soh)} across {soh['vin'].nunique()} VINs")

fig, ax = plt.subplots(figsize=(11, 5))
for vin, g in soh.groupby("vin"):
    ax.plot(g["month"], g["soh"], marker="o", ms=4, label=vin[-6:])
    ax.plot(g["month"], g["soh_raw"], ls=":", lw=0.8, alpha=0.4, color=ax.lines[-1].get_color())
ax.set_title("Mahindra SoH by coulomb counting (intellicar) — solid = monotonic, dotted = raw")
ax.set_ylabel("SoH (% of early-life capacity)"); ax.set_xlabel("Month")
ax.grid(alpha=0.3); ax.legend(title="VIN tail", ncol=2, fontsize=8)
plt.tight_layout(); plt.show()

## 5. Summary and save

In [ ]:
rows = []
for vin, g in soh.groupby("vin"):
    g = g.sort_values("month")
    yrs = max((g["month"].iloc[-1] - g["month"].iloc[0]).days / 365.25, 1e-9)
    drop = g["soh"].iloc[0] - g["soh"].iloc[-1]
    rows.append({"vin": vin,
                 "base_capacity_ah": round(float(g["capacity_ah"].iloc[0]), 1),
                 "soh_current_pct": round(float(g["soh"].iloc[-1]), 1),
                 "soh_drop_pct": round(float(drop), 1),
                 "deg_pct_per_year": round(float(drop / yrs), 2),
                 "n_points": len(g)})
summary = pd.DataFrame(rows).sort_values("soh_current_pct")
print(summary.to_string(index=False))
soh.to_parquet(SOH_OUT / "intellicar_coulomb_soh_monthly.parquet", index=False)
summary.to_csv(SOH_OUT / "intellicar_coulomb_soh_summary.csv", index=False)
print(f"\nSaved -> {SOH_OUT}/")